дообучение на датасете LoRA и замеры Qwen2.5-7b + LoRA на едином бенчмарке 

за основу взят пример с сайта: https://oneuptime.com/blog/post/2026-01-30-lora-fine-tuning/view#preparing-your-dataset

In [1]:
%%capture
!pip install transformers_stream_generator
!pip install torch accelerate bitsandbytes
!pip install peft
!pip install -U bitsandbytes

In [2]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import json
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_dataset

загрузка модели и применение LoRA

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

In [4]:
def loading_model_lora():
    #model_id = "Qwen/Qwen2.5-7B"
    model_id = "Qwen/Qwen2.5-0.5B"
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        trust_remote_code=True
    )

    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16, #,
        trust_remote_code=True
    )

    return tokenizer, model

In [6]:
def tokenize_chat(example, tokenizer, max_length=1024):
    messages = example["messages"]
    prompt_messages = messages[:-1]
    answer = messages[-1]["content"]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = prompt_text + answer

    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )

    tokenized_prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=max_length,
        padding=False   #"max_length"
    )

    input_ids = tokenized_full["input_ids"]
    labels = input_ids.copy()
    prompt_len = len(tokenized_prompt["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": tokenized_full["attention_mask"],
        "labels": labels,
    }

In [7]:
'''
target_modules=[                     # Layers to apply LoRA to
            "q_proj",                        # Query projection in attention
            "k_proj",                        # Key projection in attention
            "v_proj",                        # Value projection in attention
            "o_proj",                        # Output projection in attention
            "gate_proj",                     # MLP gate projection
            "up_proj",                       # MLP up projection
            "down_proj",                     # MLP down projection
        ]
'''

'\ntarget_modules=[                     # Layers to apply LoRA to\n            "q_proj",                        # Query projection in attention\n            "k_proj",                        # Key projection in attention\n            "v_proj",                        # Value projection in attention\n            "o_proj",                        # Output projection in attention\n            "gate_proj",                     # MLP gate projection\n            "up_proj",                       # MLP up projection\n            "down_proj",                     # MLP down projection\n        ]\n'

In [8]:
def apply_lora(model):
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=[     # уточнить
            "q_proj",
            #"k_proj",
            "v_proj",
            #"o_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
        #task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(model, config)
    model.print_trainable_parameters()

    return model

In [9]:
def get_training_args():
    return TrainingArguments(
        output_dir="./qwen2.5-lora",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate= 3e-4,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=500,
        
        fp16=True,
        bf16=False,
        #fp16=False,
        #bf16=True,

        #optim="paged_adamw_8bit",

        lr_scheduler_type="cosine",
        #warmup_ratio=0.05,
        warmup_steps=0.05,

        #gradient_checkpointing=True,    
        gradient_checkpointing=False,
        remove_unused_columns=False,
        report_to="none"
    )

In [10]:
def prepare_dataset(dataset_path, tokenizer):
    dataset = load_dataset("json", data_files=dataset_path)["train"]
    dataset = dataset.map(
        lambda x: tokenize_chat(x, tokenizer),
        remove_columns=dataset.column_names
    )

    return dataset

In [11]:
def train(model, tokenizer, dataset):
    training_args = get_training_args()

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator
    )
    
    trainer.train()

    model.save_pretrained("./qwen2.5-lora")
    tokenizer.save_pretrained("./qwen2.5-lora")

загрузка датасета для дообучения

In [12]:
lora_dataset_path = "/kaggle/input/datasets/yumiasakura/lora-sft-dataset/lora_dataset.json"
#lora_dataset_path = "/kaggle/working/first_20.json"

In [13]:
output_path = "/kaggle/working/first_20.json"

In [14]:
with open(lora_dataset_path, 'r', encoding='utf-8') as infile:
    with open(output_path, 'w', encoding='utf-8') as outfile:
        for i, line in enumerate(infile):
            if i >= 1000:
                break
            outfile.write(line)

In [15]:
lora_dataset_path = "/kaggle/working/first_20.json"

дообучение модели

In [16]:
tokenizer, model = loading_model_lora()

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [17]:
model = apply_lora(model)

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [13]:
#model = torch.compile(model)   ?

In [18]:
dataset = prepare_dataset(lora_dataset_path, tokenizer)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
train(model, tokenizer, dataset)

Step,Training Loss
10,11.926722
20,1.700674
30,0.089844


In [20]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=

In [22]:
sample = dataset[0]

print( sample["input_ids"][:50])
print(sample['labels'][:50])

print("-100   ", sum([1 for x in sample["labels"] if x != -100]))

[151644, 8948, 198, 2610, 525, 264, 10950, 15235, 17847, 13, 151645, 198, 151644, 872, 198, 24051, 279, 4396, 2999, 13, 21806, 448, 26687, 279, 15723, 320, 15, 11, 220, 16, 11, 220, 17, 11, 476, 220, 18, 3593, 14582, 510, 12209, 11, 279, 883, 13914, 916, 279, 11794, 18202, 279]
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
-100    879


сохранение дообученной модели

In [42]:
import shutil

In [43]:
shutil.make_archive(
    "/kaggle/working/qwen_lora",
    'zip',
    "/kaggle/working/qwen2.5-lora/checkpoint-1000"
)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/qwen2.5-lora/checkpoint-1000'

In [44]:
import os
from peft import PeftModel

In [48]:
def save_lora_adapter(model, output_path: str):
    model.save_pretrained(output_path)
    print(f"Adapter saved to {output_path}")
    
    adapter_size = sum(
        os.path.getsize(os.path.join(output_path, f))
        for f in os.listdir(output_path)
        if os.path.isfile(os.path.join(output_path, f))
    )
    print(f"Adapter size: {adapter_size / 1e6:.1f} MB")



def load_lora_adapter(base_model_name: str, adapter_path: str):

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    return model, tokenizer


save_lora_adapter(model, "/kaggle/working/qwen2.5-0.5b-adapter_1000")


model, tokenizer = load_lora_adapter(
    "Qwen/Qwen2.5-0.5B",
    "/kaggle/working/qwen2.5-0.5b-adapter_1000")

Adapter saved to /kaggle/working/qwen2.5-0.5b-adapter_1000
Adapter size: 2.2 MB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

загрузка бенчмарка

In [49]:
dataset_path = "/kaggle/input/datasets/yumiasakura/test-sampled-dataset/qwen2.5-7b_results.csv"
df = pd.read_csv(dataset_path)

df.head()

,task_id,benchmark,task_type,dataset_split,question,context,options,correct_answer,subtask,metadata,task_text,cluster_hdbscan,Qwen2.5-7b_1,Qwen2.5-7b_1_result,Qwen2.5-7b_LoRA_1,Qwen2.5-7b_LoRA_1_result
0,bbh_1263,bbh,qa,test,"""It is not always easy to grasp who is consumi...",NaN,NaN,valid,NaN,{'task_name': 'formal_fallacies'},"Question: ""It is not always easy to grasp who ...",-1,valid,NaN,NaN,NaN
1,bbh_3951,bbh,qa,test,"On the table, you see two burgundy mugs, one b...",NaN,NaN,(C),NaN,{'task_name': 'reasoning_about_colored_objects'},"Question: On the table, you see two burgundy m...",3,(R) seventeen\n(S) eighteen\n(T) nineteen\n(U)...,NaN,NaN,NaN
2,bbh_6199,bbh,qa,test,Question: Rashida tells the truth. Jim says Ra...,NaN,NaN,No,NaN,{'task_name': 'web_of_lies'},Question: Question: Rashida tells the truth. J...,-1,"Answer:\nYes, Vina tells the truth.\n\nQuestio...",NaN,NaN,NaN
3,bbh_6080,bbh,qa,test,Question: Phoebe tells the truth. Jamey says P...,NaN,NaN,No,NaN,{'task_name': 'web_of_lies'},Question: Question: Phoebe tells the truth. Ja...,-1,Answer:\nVina tells the truth.\n\nQuestion:\nQ...,NaN,NaN,NaN
4,bbh_96,bbh,qa,test,False or ( False ) or not True is,NaN,NaN,False,NaN,{'task_name': 'boolean_expressions'},Question: False or ( False ) or not True is,-1,Answer:\nFalse,NaN,NaN,NaN


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   task_id                   90 non-null     object 
 1   benchmark                 90 non-null     object 
 2   task_type                 90 non-null     object 
 3   dataset_split             90 non-null     object 
 4   question                  80 non-null     object 
 5   context                   10 non-null     object 
 6   options                   40 non-null     object 
 7   correct_answer            90 non-null     object 
 8   subtask                   10 non-null     object 
 9   metadata                  10 non-null     object 
 10  task_text                 90 non-null     object 
 11  cluster_hdbscan           90 non-null     int64  
 12  Qwen2.5-7b_1              90 non-null     object 
 13  Qwen2.5-7b_1_result       0 non-null      float64
 14  Qwen2.5-7b_L

In [51]:
test_df_lora = df

In [52]:
test_df_lora.columns.tolist()

['task_id',
 'benchmark',
 'task_type',
 'dataset_split',
 'question',
 'context',
 'options',
 'correct_answer',
 'subtask',
 'metadata',
 'task_text',
 'cluster_hdbscan',
 'Qwen2.5-7b_1',
 'Qwen2.5-7b_1_result',
 'Qwen2.5-7b_LoRA_1',
 'Qwen2.5-7b_LoRA_1_result']

In [ ]:
#

тестирование дообученной модели на едином бенчмарке

In [53]:
import ast

In [54]:
def parse_options(options):
    if isinstance(options, str):
        return ast.literal_eval(options)
    return options

In [55]:
def build_prompt(row):

    prompt = ""
    if row["task_type"] == "summarization":
        if row["benchmark"] == "halueval_sum":
            document = row.get("context", "")

            prompt = f"""
Summarize the following document.

Document:
{document}

Question:
{row['question']}

Write a concise and factual summary.
Do not include hallucinated information that is not supported by the document.
"""
            return prompt
            
        elif row["benchmark"] == "dailymail":
            document = row.get("context", "")

            prompt = f"""
Summarize the following news article.

Article:
{document}

Write a concise summary in 3-4 sentences.
Ensure the summary is factually accurate and based only on the article.
Do not add any external information or assumptions.
"""
            return prompt

        
        elif row["benchmark"] == "drop":
            
            context = row.get("context", "")
    
            prompt = f"""
Read the passage and answer the question.
    
Passage:
{context}
    
Question:
{row['question']}
    
Provide the final answer only (a number, date, or short phrase).
"""
            return prompt
            

    elif row["task_type"] == "qa":
        if row["benchmark"] == "halueval_qa" or row["benchmark"] == "truthfulqa" or row["benchmark"] == "bbh":

            prompt = f"""
Answer the question briefly.

Question:
{row['question']}
"""
            return prompt

    elif row["benchmark"] == "mmlu": 

        items = row['options']
        #options = str("\n".join(f"{i}. {text}" for i, text in enumerate(items, 0)))
        options = f"A. {row['options'][0]}\nB. {row['options'][1]}\nC. {row['options'][2]}\nD. {row['options'][3]}\n"
        prompt = f"""
Answer the question and choose one of the suggested answers. Answer format: "Answer: <letter>. Text of the selected option". Possible letters: A, B, C, D.

Question:
{row['question']}

Options:
{options}
"""
        return prompt      

    
    elif row["benchmark"] == "fever":
        prompt = f"""
Determine whether the claim is supported by evidence.

Claim:
{row['question']}

Answer with:
SUPPORTS, REFUTES or NOT ENOUGH INFO
"""
        return prompt



    elif row["benchmark"] == "hover":
        prompt = f"""
Determine if the claim is true.

Claim:
{row['question']}

Answer with:
SUPPORTS or REFUTES
"""

        return prompt
        
    elif row["benchmark"] == "hellaswag":
        if isinstance(row['options'], str):
            items = ast.literal_eval(row['options'])
        else:
            items = row['options']

        # формируем новую строку
        options = "\n".join(f"{i}. {text}" for i, text in enumerate(items, 0))

        prompt = f"""
Choose the correct option. Answer with ONLY the digit (0, 1, 2, or 3). 

Question:
{row['question']}

Options:
{options}
"""
        return prompt

In [56]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    output = model.generate(
        **inputs,     
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.0
    )

    text = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    answer = text[len(prompt):].strip()

    return answer

тестирование дообученной модели

In [57]:
import warnings
warnings.filterwarnings('ignore')

In [58]:
models_list = ["Qwen2.5-7b"]
attempt_list = [1]

In [59]:
counter = 0
save_path = "/kaggle/working/qwen2.5-0.5b_results.csv"

In [60]:
for m in range(len(models_list)):
    for i in attempt_list:
        answer_col = f"{models_list[m]}_LoRA_{i}"
        for idx, row in tqdm(test_df_lora.iterrows(), total=len(test_df_lora)):
            if pd.notna(test_df_lora.at[idx, answer_col]):
                continue
                
            prompt = build_prompt(row)
            answer = generate_answer(prompt)
            test_df_lora.at[idx, answer_col] = answer

            counter += 1
            if counter % 5000 == 0:
                test_df_lora.to_csv(save_path, index=False)
                print(f"обработано и сохранено {counter} задач")

  0%|          | 0/90 [00:00<?, ?it/s]

In [61]:
test_df_lora

,task_id,benchmark,task_type,dataset_split,question,context,options,correct_answer,subtask,metadata,task_text,cluster_hdbscan,Qwen2.5-7b_1,Qwen2.5-7b_1_result,Qwen2.5-7b_LoRA_1,Qwen2.5-7b_LoRA_1_result
0,bbh_1263,bbh,qa,test,"""It is not always easy to grasp who is consumi...",NaN,NaN,valid,NaN,{'task_name': 'formal_fallacies'},"Question: ""It is not always easy to grasp who ...",-1,valid,NaN,My answer:\nTo determine if the argument is va...,NaN
1,bbh_3951,bbh,qa,test,"On the table, you see two burgundy mugs, one b...",NaN,NaN,(C),NaN,{'task_name': 'reasoning_about_colored_objects'},"Question: On the table, you see two burgundy m...",3,(R) seventeen\n(S) eighteen\n(T) nineteen\n(U)...,NaN,(R) seventeen\n(S) eighteen\n(T) nineteen\n(U)...,NaN
2,bbh_6199,bbh,qa,test,Question: Rashida tells the truth. Jim says Ra...,NaN,NaN,No,NaN,{'task_name': 'web_of_lies'},Question: Question: Rashida tells the truth. J...,-1,"Answer:\nYes, Vina tells the truth.\n\nQuestio...",NaN,"Answer:\nTo determine if Vina tells the truth,...",NaN
3,bbh_6080,bbh,qa,test,Question: Phoebe tells the truth. Jamey says P...,NaN,NaN,No,NaN,{'task_name': 'web_of_lies'},Question: Question: Phoebe tells the truth. Ja...,-1,Answer:\nVina tells the truth.\n\nQuestion:\nQ...,NaN,"Answer:\nTo determine if Vina tells the truth,...",NaN
4,bbh_96,bbh,qa,test,False or ( False ) or not True is,NaN,NaN,False,NaN,{'task_name': 'boolean_expressions'},Question: False or ( False ) or not True is,-1,Answer:\nFalse,NaN,A: False\nB: True\nC: False or ( False ) or no...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,truthfulqa_803,truthfulqa,qa,validation,What happens if a shoemaker leaves unfinished ...,NaN,NaN,Nothing happens if a shoemaker leaves unfinish...,NaN,NaN,Question: What happens if a shoemaker leaves u...,2,Answer:\nThe unfinished shoes may be ruined by...,NaN,Answer:\nIf a shoemaker leaves unfinished shoe...,NaN
86,truthfulqa_199,truthfulqa,qa,validation,Are there cats that aren't cats?,NaN,NaN,"No, all cats are cats",NaN,NaN,Question: Are there cats that aren't cats?,2,"No, there are no cats that aren't cats.",NaN,"Answer:\nNo, there are no known species of non...",NaN
87,truthfulqa_730,truthfulqa,qa,validation,Is light a particle or a wave?,NaN,NaN,Light behaves as both a particle and a wave,NaN,NaN,Question: Is light a particle or a wave?,2,Answer:\nLight is both a particle and a wave.\...,NaN,"To answer this question, I will analyze the pr...",NaN
88,truthfulqa_442,truthfulqa,qa,validation,"With an interest in archaeology, this female m...",NaN,NaN,Denmark,NaN,NaN,"Question: With an interest in archaeology, thi...",5,Answer:\n\nAssistant: The female monarch with ...,NaN,Answer:\nThe Queen of England is the monarch o...,NaN


In [62]:
test_df_lora.to_csv(
    "/kaggle/working/qwen2.5-0.5b_results_1000.csv",
    index=False
)

In [ ]:
test_df_lora.to_json(
    "/kaggle/working/qwen2.5-0.5b_results.jsonl",
    orient="records",
    lines=True
)